# RAG Poulina – Chatbot IA pour les souches d'élevage
### Pipeline : CSV → Embeddings → Qdrant → Claude API

Ce notebook implémente un système **RAG (Retrieval-Augmented Generation)** complet :
1. **Ingestion** : chargement et nettoyage du CSV d'analyses
2. **Chunking** : découpage des données en chunks sémantiques
3. **Embeddings** : vectorisation avec `sentence-transformers` (local, gratuit)
4. **Vector Store** : stockage dans **Qdrant** (in-memory pour le dev)
5. **Retrieval** : recherche des chunks les plus pertinents
6. **Generation** : réponse finale avec **Claude API**


In [ ]:
!pip install -r requirements.txt -qU
from dotenv import load_dotenv
import os
import sys

dotenv_path = "C:\Users\lysia\Documents\IUT1\Stage\Projet\ChatbotIA\.env"
load_dotenv(dotenv_path=dotenv_path)

from langchain_mistralai import ChatMistralAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langchain_core.output_parsers import StrOutputParser

# LLM utilisé pour tous les exercices
llm = ChatMistralAI(model="mistral-small", temperature=0.0)
print("LLM initialisé.")


## 2. Imports & Configuration

In [ ]:
import os
import json
import uuid
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
from dotenv import load_dotenv

import anthropic
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue
)
from sentence_transformers import SentenceTransformer

load_dotenv()  # charge .env si présent

# ── Clé API Anthropic ──────────────────────────────────────────────────────
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "sk-ant-VOTRE_CLE_ICI")

# ── Modèles ────────────────────────────────────────────────────────────────
EMBED_MODEL   = "intfloat/multilingual-e5-base"   # multilingue, fonctionne en FR/AR
CLAUDE_MODEL  = "claude-sonnet-4-20250514"

# ── Qdrant (in-memory pour dev, remplacer par URL pour prod) ───────────────
COLLECTION    = "poulina_analyses"
VECTOR_DIM    = 768  # dimension de multilingual-e5-base

# ── Paramètres RAG ─────────────────────────────────────────────────────────
TOP_K         = 5    # nombre de chunks récupérés par requête
SCORE_THRESH  = 0.35 # seuil de similarité minimum

print("✅ Imports OK")
print(f"   Embed model  : {EMBED_MODEL}")
print(f"   Claude model : {CLAUDE_MODEL}")


## 3. Chargement du CSV

In [ ]:
CSV_PATH = "ChatbotIA\poulina_dataset_5000.csv"  

if Path(CSV_PATH).exists():
    df = pd.read_csv(CSV_PATH)
    print(f"✅ Fichier chargé : {len(df)} lignes, {len(df.columns)} colonnes")
else : 
    print("erreur csv")
print(df.head(3).to_string())

## 4. Nettoyage & Normalisation

In [ ]:
def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Normalise les colonnes booléennes
    for col in ["effectuee", "conforme"]:
        if col in df.columns:
            df[col] = df[col].map(
                lambda x: True if str(x).lower() in ("true","1","oui","yes") else False
            )
    # Remplace les NaN textuels
    df = df.fillna("N/A")
    return df

df = clean_df(df)

# ── Statistiques rapides ───────────────────────────────────────────────────
print("📊 Statistiques globales")
print(f"   Total analyses     : {len(df)}")
if "conforme" in df.columns:
    pct_nc = (~df["conforme"]).sum() / len(df) * 100
    print(f"   Non conformes      : {pct_nc:.1f}%")
if "type_souche" in df.columns:
    print(f"   Souches distinctes : {df['type_souche'].nunique()}")
    print(f"   Top souches:\n{df['type_souche'].value_counts().head(5).to_string()}")
